# =================================================================
# PRODUCTION TARBALL BUILDER: Tagged LuaLaTeX (PDF/UA-2 / ISO 32005)
# Designed for: Looney & Duston (2026) Physics Video & STEM Publishing Framework
#
# Output: A lightweight (~140 MB) standalone TeX Live distribution optimized for
#         cloud execution, instant unpacking (~12-15s), and ISO 32005 compliance.
# =================================================================


The Github connection is going to be done with a personal access tolken - use the key to the left in the Colab interface, add new secret GITHUB_PAT, and make the value a key associated to your github -> settings > credentials > fine-grained PAT -> Generate new tolken, give it permissions for the repo, and also make sure it has read and write access to the Contents under permissions. When you get it, put it's value into the Value spot in the secrets. Make sure to give the tolken notebook access.

Naturally, this is designed for me (C Duston) to use, and would have to be adapted for a user who wants to contribute to this repo.

In [19]:
GITHUB_FLAG=1 # 0 for skipping github release creation, 1 to create
TAG="v0.0.3" # version tag for resulting github release. Note if this exists already, this github thing will fail.
TITLE=f"{TAG} -fixed tarball structure"
NOTES="Fixed the tarball structure and added github release capibility"

In [4]:
import os
import subprocess
import shutil
import time # for timing the process
_notebook_start = time.time()

In [5]:
print("=" * 70)
print("STEP 1: Downloading Official TeX Live Network Installer")
print("=" * 70)

# these commands will build things in the remote, presumably virtual, machine.
!mkdir -p /tmp/tl-build /tmp/texlive-academic-tagged
%cd /tmp/tl-build

# Fetch the official upstream TeX Live network installer
!curl -sL https://mirror.ctan.org/systems/texlive/tlnet/install-tl-unx.tar.gz | tar -xz --strip-components=1

STEP 1: Downloading Official TeX Live Network Installer
/tmp/tl-build


In [6]:
# Configure minimal, portable installation profile (docs and source files stripped)
profile_content = """
selected_scheme scheme-infraonly
TEXDIR /tmp/texlive-academic-tagged
TEXMFCONFIG ~/.texlive/texmf-config
TEXMFHOME ~/texmf
TEXMFLOCAL /tmp/texlive-academic-tagged/texmf-local
TEXMFSYSCONFIG /tmp/texlive-academic-tagged/texmf-config
TEXMFSYSVAR /tmp/texlive-academic-tagged/texmf-var
TEXMFVAR ~/.texlive/texmf-var
binary_x86_64-linux 1
instopt_adjustpath 0
instopt_adjustrepo 1
instopt_letter 0
instopt_portable 1
instopt_write18_restricted 1
tlpdbopt_autobackup 0
tlpdbopt_create_formats 1
"""

with open("texlive.profile", "w") as f:
    f.write(profile_content.strip())

In [7]:

print("\n" + "=" * 70)
print("STEP 2: Installing Base Infrastructure & LuaTeX Engine")
print("=" * 70)
!./install-tl --profile=texlive.profile

# Set local execution path for the remainder of the build
bin_dir = "/tmp/texlive-academic-tagged/bin/x86_64-linux"
os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"


STEP 2: Installing Base Infrastructure & LuaTeX Engine
Automated TeX Live installation using profile: texlive.profile
Loading https://mirror.math.princeton.edu/pub/CTAN/systems/texlive/tlnet/tlpkg/texlive.tlpdb
Installing TeX Live 2026 from: https://mirror.math.princeton.edu/pub/CTAN/systems/texlive/tlnet (verified)
Platform: x86_64-linux => 'GNU/Linux on x86_64'
Distribution: net  (downloading)
Using URL: https://mirror.math.princeton.edu/pub/CTAN/systems/texlive/tlnet
Directory for temporary files: /tmp/aSAfYBXCkf
Installing to: /tmp/texlive-academic-tagged
Installing [1/4, time/total: ??:??/??:??]: hyphen-base [23k]
Installing [2/4, time/total: 00:01/02:22]: kpathsea [1291k]
Installing [3/4, time/total: 00:02/00:04]: texlive-scripts [1190k]
Installing [4/4, time/total: 00:04/00:05]: texlive.infra [657k]
Time used for installing the packages: 00:06
Installing [1/3, time/total: ??:??/??:??]: kpathsea.x86_64-linux [41k]
Installing [2/3, time/total: 00:00/00:00]: texlive-scripts.x86_64

This next block contains errors "tlmgr: action install returned an error; continuing." which Claude says can be safely ignored.

In [8]:
print("\n" + "=" * 70)
print("STEP 3: Installing Complete Package Manifest")
print("=" * 70)

# 3A. Core Engines & Format Builders
print("--> Installing core engines and format generators...")
!tlmgr install luatex luahbtex latex-bin l3kernel l3backend l3packages

# 3B. Modern PDF/UA-2 (ISO 32005) Tagging Engine
print("--> Installing LaTeX Tagging Project & PDF 2.0 namespace modules...")
!tlmgr install latex-lab tagpdf pdfmanagement lua-uni-algos lualibs luaotfload

# 3C. Fonts & OpenType Math (NewComputerModern ecosystem)
print("--> Installing OpenType math and unicode fonts...")
!tlmgr install fontspec unicode-math lualatex-math newcomputermodern lm lm-math

# 3D. Core Preamble Packages (v6m & arXiv Paper Preamble)
print("--> Installing core layout, math, units, and hyperref...")
!tlmgr install geometry graphics graphics-def
!tlmgr install amsmath mathtools siunitx translations
!tlmgr install parskip tools hologo xurl url
!tlmgr install hyperref bookmark auxhook kvoptions etoolbox xcolor

# 3E. Vector Graphics, Diagrams & TikZ
print("--> Installing TikZ, PGFPlots, and Circuitikz...")
!tlmgr install pgf pgfplots circuitikz

# 3F. Academic Physics Safety-Net (Tables, boxes, derivations)
print("--> Installing safety-net and academic layout packages...")
!tlmgr install cancel physics diffcoeff bm
!tlmgr install booktabs array tabularx enumitem
!tlmgr install caption subcaption tcolorbox environ

# 3G. Additional packages that were missing from the original script
!tlmgr install fontsetup luamml latex-lab-math colorprofiles


STEP 3: Installing Complete Package Manifest
--> Installing core engines and format generators...
tlmgr: package repository https://mirror.math.princeton.edu/pub/CTAN/systems/texlive/tlnet (verified)
tlmgr install: package l3backend not present in repository.
[1/37, ??:??/??:??] install: amsmath [2431k]
[2/37, 00:03/01:59] install: babel [2358k]
[3/37, 00:06/02:01] install: cm [235k]
[4/37, 00:07/02:14] install: dehyph [46k]
[5/37, 00:07/02:13] install: epstopdf.x86_64-linux [1k]
[6/37, 00:08/02:32] install: epstopdf [44k]
[7/37, 00:09/02:50] install: epstopdf-pkg [362k]
[8/37, 00:10/02:56] install: etex [331k]
[9/37, 00:11/03:03] install: firstaid [298k]
[10/37, 00:13/03:25] install: graphics [2462k]
[11/37, 00:15/02:49] install: graphics-cfg [2k]
[12/37, 00:16/03:00] install: graphics-def [12k]
[13/37, 00:17/03:11] install: hyph-utf8 [313k]
[14/37, 00:19/03:26] install: knuth-lib [30k]
[15/37, 00:20/03:36] install: l3kernel [12792k]
[16/37, 00:23/01:42] install: l3packages [1431k]
[

In [9]:
print("\n" + "=" * 70)
print("STEP 4: Smoke Test (Compiling Tagged Document with Math, Units & TikZ)")
print("=" * 70)

smoke_test_tex = r"""
\DocumentMetadata{
	pdfstandard = {ua-2, a-4f},
	lang = en-US,
	tagging = on,
	tagging-setup = {math/setup = {mathml-SE}}
}

\documentclass[11pt, letterpaper]{article}
\usepackage[margin=1in]{geometry}

\usepackage{amsmath}
\usepackage{newcomputermodern}
\usepackage{siunitx}
\usepackage{graphicx}
\usepackage{parskip}
\usepackage{alltt}
\usepackage{hologo}
\usepackage{tikz}

\providecommand{\square}{\mdlgwhtsquare}

\sisetup{
	locale = US,
	separate-uncertainty = true,
	tight-spacing = true,
	per-mode = symbol
}

\usepackage{xcolor}
\definecolor{arxivblue}{rgb}{0.1, 0.2, 0.6}

\usepackage{hyperref}
\hypersetup{
	pdfdisplaydoctitle = true,
	colorlinks=true,
	linkcolor=black,
	citecolor=arxivblue,
	urlcolor=arxivblue,
	breaklinks=true
}
\usepackage{xurl}

\title{Tagged LuaLaTeX Compiler Smoke Test}
\author{Looney \& Duston Framework Engine}
\date{\today}

\begin{document}
\maketitle

\begin{abstract}
Automated verification test for the minimal production-ready tagged LuaLaTeX compiler distribution.
\end{abstract}

\section{Accessible Derivations, Units \& Diagrams}
Testing SI units: \qty{3.00e8}{\meter\per\second}, engine logo: \hologo{LuaLaTeX}, and structural math tagging:
\begin{equation}
i\hbar \frac{\partial}{\partial t}\Psi(\mathbf{r}, t) = \left[ -\frac{\hbar^2}{2m}\nabla^2 + V(\mathbf{r}, t) \right] \Psi(\mathbf{r}, t)
\end{equation}

Testing vector diagram generation (TikZ):
\begin{center}
\begin{tikzpicture}
    \draw[thick, ->] (0,0) -- (3,0) node[anchor=north] {$x$};
    \draw[thick, ->] (0,0) -- (0,3) node[anchor=east] {$y$};
    \draw[red, thick, ->] (0,0) -- (2,2) node[anchor=south west] {$\mathbf{F}_{\text{net}}$};
\end{tikzpicture}
\end{center}

\end{document}
"""

with open("/tmp/production_smoke_test.tex", "w") as f:
    f.write(smoke_test_tex.strip())

# Execute dual-pass compilation
result1 = subprocess.run(["lualatex", "--interaction=nonstopmode", "/tmp/production_smoke_test.tex"],
                         cwd="/tmp", capture_output=True, text=True)
result2 = subprocess.run(["lualatex", "--interaction=nonstopmode", "/tmp/production_smoke_test.tex"],
                         cwd="/tmp", capture_output=True, text=True)

if result2.returncode == 0:
    print(" SUCCESS: Test document compiled with exit code 0 and verified PDF/UA-2 tagging!")
else:
    print(" FAILED: Compilation Errors detected:")
    print(result2.stdout[-1500:])
    raise RuntimeError("Compilation test failed. Check missing dependencies.")


STEP 4: Smoke Test (Compiling Tagged Document with Math, Units & TikZ)
 SUCCESS: Test document compiled with exit code 0 and verified PDF/UA-2 tagging!


In [10]:
print("\n" + "=" * 70)
print("STEP 5: Compressing Production Tarball")
print("=" * 70)

# Clean up log files before packaging
!rm -rf /tmp/texlive-academic-tagged/texmf-var/web2c/tlmgr.log
%cd /tmp

tarball_name = "texlive-physics-tagged.tar.gz"
#!tar -czf {tarball_name} -C /tmp texlive-academic-tagged

%cd /tmp
!tar -czf {tarball_name} -C /tmp/texlive-academic-tagged .

print(f"\n COMPLETE! Production archive generated: /tmp/{tarball_name}")
!ls -lh /tmp/{tarball_name}


STEP 5: Compressing Production Tarball
/tmp
/tmp

 COMPLETE! Production archive generated: /tmp/texlive-physics-tagged.tar.gz
-rw-r--r-- 1 root root 271M Sep  4 21:11 /tmp/texlive-physics-tagged.tar.gz


In [11]:
# Optional automatic download if running in Google Colab:
try:
    from google.colab import files
    print("\nTriggering browser download...")
    files.download(f"/tmp/{tarball_name}")
except ImportError:
    print(f"\nRunning outside Colab. File is saved at: /tmp/{tarball_name}")

# Let's also grab the test .pdf
from google.colab import files
files.download('/tmp/production_smoke_test.pdf')


Triggering browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Just a note, if you want to stick on github, create a release and stick the .tar.gz there, it will be too big for a normal upload!

In [12]:
elapsed = time.time() - _notebook_start
mins, secs = divmod(elapsed, 60)
print(f"Total notebook runtime: {int(mins)}m {secs:.1f}s")

Total notebook runtime: 5m 1.6s


In [13]:
!ls /tmp

initgoogle_syslog_dir.0
language_service.9d74c5ae6149.root.log.INFO.20260904-191246.835
language_service.9d74c5ae6149.root.log.INFO.20260904-203332.21265
language_service.9d74c5ae6149.root.log.INFO.20260904-210504.32608
language_service.INFO
production_smoke_test.aux
production_smoke_test.log
production_smoke_test-luamml-mathml.html
production_smoke_test.pdf
production_smoke_test.tex
pyright-32615-L4xfstKywk1M
python-languageserver-cancellation
{tarball_name}
texlive-academic-tagged
texlive-physics-tagged.tar.gz
tl-build


In [14]:
!tar -tzf {tarball_name} | head -20

./
./bin/
./bin/x86_64-linux/
./bin/x86_64-linux/etex
./bin/x86_64-linux/tlmgr
./bin/x86_64-linux/updmap
./bin/x86_64-linux/dvilualatex
./bin/x86_64-linux/kpsereadlink
./bin/x86_64-linux/latex
./bin/x86_64-linux/man
./bin/x86_64-linux/mktexmf
./bin/x86_64-linux/kpsewhich
./bin/x86_64-linux/updmap-user
./bin/x86_64-linux/simpdftex
./bin/x86_64-linux/texhash
./bin/x86_64-linux/dviluatex
./bin/x86_64-linux/fmtutil-sys
./bin/x86_64-linux/kpseaccess
./bin/x86_64-linux/lualatex
./bin/x86_64-linux/kpsestat


In [15]:
!tar -tzf {tarball_name} | grep "bin/x86_64-linux/lualatex"

./bin/x86_64-linux/lualatex


Now we want some code that will send this and other updates to github. Check the locations first...

In [18]:
!ls {tarball_name}

texlive-physics-tagged.tar.gz


In [23]:
if GITHUB_FLAG==1:
  # One-time setup in this runtime
  !curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg | dd of=/usr/share/keyrings/githubcli-archive-keyring.gpg
  !echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main" | tee /etc/apt/sources.list.d/github-cli.list > /dev/null
  !apt-get update -qq && apt-get install -y gh

  from google.colab import userdata
  import os
  os.environ["GH_TOKEN"] = userdata.get('GITHUB_PAT')  # never printed, never in cell output

  # Create the release and upload the asset in one command
  !gh release create {TAG} {tarball_name} --repo cduston44/mini-lualatex --title "{TITLE}" --notes "{NOTES}"

8+1 records in
8+1 records out
4528 bytes (4.5 kB, 4.4 KiB) copied, 0.128768 s, 35.2 kB/s
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gh is already the newest version (2.100.0).
0 upgraded, 0 newly installed, 0 to remove and 79 not upgraded.
https://github.com/cduston44/mini-lualatex/releases/tag/v0.0.3
